# Notebook 2 — Feature Engineering
**Geotechnical Shear Strength Predictive Suite**

Pipeline: Load cleaned data → Liquidity Index → Synthetic CPT resistance → Correlation heatmap → Export

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
INPUT_PATH       = 'geotechnical_cleaned.csv'
OUTPUT_PATH      = 'geotechnical_engineered.csv'
SPT_TO_QC_FACTOR = 0.4
LI_EPSILON       = 0.001
HEATMAP_FEATURES = ['LL', 'PL', 'PI', 'Wn_%', 'LI', 'qc_est', 'is_plastic', 'Phi_deg', 'Cu_kPa']

## 1 · Load Cleaned Data

In [ ]:
def load_cleaned(path):
    # Read cleaned CSV produced by Notebook 1
    df = pd.read_csv(path)
    print(f'Loaded {df.shape[0]} rows × {df.shape[1]} columns')
    return df

In [ ]:
df = load_cleaned(INPUT_PATH)
df[['LL', 'PL', 'PI', 'Wn_%', 'SPT_N', 'CPT_MPa', 'is_plastic', 'Phi_deg', 'Cu_kPa']].head(6)

## 2 · Liquidity Index (LI)

$$LI = \frac{W_n - PL}{PI + \varepsilon}$$

- $\varepsilon = 0.001$ prevents division-by-zero for Non-Plastic (NP) soils where $PI = 0$.
- `LI` will be `NaN` where `Wn_%` is missing — these rows are usable in models that tolerate partial feature availability.

In [ ]:
def compute_liquidity_index(df, epsilon):
    # LI measures proximity of natural water content to plastic/liquid limits
    df = df.copy()
    df['LI'] = (df['Wn_%'] - df['PL']) / (df['PI'] + epsilon)
    non_null = df['LI'].notna().sum()
    null_count = df['LI'].isna().sum()
    print(f'LI computed: {non_null} values, {null_count} NaN (Wn_% missing)')
    print(f'LI range: [{df["LI"].min():.3f}, {df["LI"].max():.3f}]')
    return df

In [ ]:
df = compute_liquidity_index(df, LI_EPSILON)
df[['Wn_%', 'PL', 'PI', 'LI']].dropna().head(6)

## 3 · Synthetic Resistance — `qc_est`

CPT data (`CPT_MPa`) is entirely absent from this dataset. Where SPT blow count
(`SPT_N`) is available, a synthetic cone resistance is estimated:

$$q_c \approx 0.4 \times N_{SPT}$$

Rows with neither CPT nor SPT remain `NaN` in `qc_est`.

In [ ]:
def compute_qc_est(df, factor):
    # Initialise qc_est from CPT values where available
    df = df.copy()
    df['qc_est'] = df['CPT_MPa'].copy()
    # Apply SPT fallback only when CPT is null and SPT is present
    spt_fallback_mask = df['CPT_MPa'].isna() & df['SPT_N'].notna()
    df.loc[spt_fallback_mask, 'qc_est'] = factor * df.loc[spt_fallback_mask, 'SPT_N']
    cpt_used  = df['CPT_MPa'].notna().sum()
    spt_used  = spt_fallback_mask.sum()
    null_left = df['qc_est'].isna().sum()
    print(f'qc_est filled from CPT: {cpt_used}')
    print(f'qc_est filled from SPT (×{factor}): {spt_used}')
    print(f'qc_est remaining NaN: {null_left}')
    print(f'qc_est range: [{df["qc_est"].min():.2f}, {df["qc_est"].max():.2f}]')
    return df

In [ ]:
df = compute_qc_est(df, SPT_TO_QC_FACTOR)
df[['SPT_N', 'CPT_MPa', 'qc_est']].dropna(subset=['qc_est']).head(6)

## 4 · Correlation Heatmap
Pearson correlation matrix for the 7 engineered features plus both targets (`Phi_deg`, `Cu_kPa`).

In [ ]:
def compute_correlation_matrix(df, cols):
    # Compute Pearson correlation on available columns only
    available = [c for c in cols if c in df.columns]
    corr = df[available].corr(method='pearson')
    return corr

In [ ]:
def plot_correlation_heatmap(corr, targets):
    # Draw annotated heatmap and highlight target columns
    fig, ax = plt.subplots(figsize=(10, 8))
    # Use diverging palette centred at zero
    cmap = sns.diverging_palette(220, 20, as_cmap=True)
    sns.heatmap(
        corr,
        annot=True,
        fmt='.2f',
        cmap=cmap,
        center=0,
        linewidths=0.5,
        linecolor='#cccccc',
        square=True,
        annot_kws={'size': 9},
        ax=ax
    )
    ax.set_title('Feature Correlation Matrix — incl. Targets (Phi_deg, Cu_kPa)',
                 fontsize=13, fontweight='bold', pad=14)
    # Bold tick labels for target columns
    for tick in ax.get_xticklabels():
        if tick.get_text() in targets:
            tick.set_fontweight('bold')
    for tick in ax.get_yticklabels():
        if tick.get_text() in targets:
            tick.set_fontweight('bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Heatmap saved → correlation_heatmap.png')

In [ ]:
corr_matrix = compute_correlation_matrix(df, HEATMAP_FEATURES)
plot_correlation_heatmap(corr_matrix, targets=['Phi_deg', 'Cu_kPa'])

### 4.1 · Target Correlation Summary

In [ ]:
def print_target_correlations(corr, targets):
    # Print sorted correlation of all features against each target
    for target in targets:
        if target in corr.columns:
            target_corr = corr[target].drop(labels=targets).sort_values(key=abs, ascending=False)
            print(f'Correlation with {target}:')
            print(target_corr.round(3).to_string())
            print()

In [ ]:
print_target_correlations(corr_matrix, targets=['Phi_deg', 'Cu_kPa'])

## 5 · Final Shape Check

In [ ]:
def summarise_engineered(df):
    # Report new feature counts and null summaries
    new_cols = ['LI', 'qc_est']
    print(f'Final shape: {df.shape}')
    print()
    print('Engineered feature null summary:')
    print(df[new_cols].isnull().sum().to_string())
    print()
    print('Engineered feature statistics:')
    print(df[new_cols].describe().round(3).to_string())

In [ ]:
summarise_engineered(df)
df[['LL','PL','PI','Wn_%','LI','qc_est','is_plastic','Phi_deg','Cu_kPa']].head(8)

## 6 · Export

In [ ]:
def export_csv(df, path):
    # Write engineered DataFrame to CSV without row index
    df.to_csv(path, index=False)
    print(f'Exported → {path}  ({df.shape[0]} rows × {df.shape[1]} cols)')

In [ ]:
export_csv(df, OUTPUT_PATH)